In [2]:
import os
import nilearn
import nibabel as nib
import matplotlib.pyplot as plt
from nilearn import plotting
import numpy as np
from nilearn.glm.first_level import FirstLevelModel, make_first_level_design_matrix
from nilearn.plotting import plot_contrast_matrix
from nilearn.image import resample_img
from nilearn import masking
from nilearn.image import new_img_like

from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import KFold
from tqdm import tqdm

In [3]:
single_trial_betas = np.load("./data/reordered_singletrial_betas.npy")
single_trial_betas.shape

(7, 600, 79, 95, 79)

In [ ]:
# single_trial_z = np.load("./data/reordered_singletrial_z.npy")
# single_trial_z.shape

In [4]:
data = single_trial_betas

In [5]:
conditions = ['pleasant', 'neutral', 'unpleasant', 'pleasantAI', 'neutralAI', 'unpleasantAI']

# Self-decoding

In [6]:
# Specify the two conditions you want to use for decoding
condition1 = 'pleasant'
condition2 = 'neutral'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]

    # Calculate z-score for condition1_trials and condition2_trials
    # condition1_trials = zscore(condition1_trials, axis=1)
    # condition2_trials = zscore(condition2_trials, axis=1)

    for repeat in tqdm(range(20)):
        kf = KFold(n_splits=4, shuffle=True, random_state=repeat)
        condition1_indices = np.arange(condition1_trials.shape[0])
        condition2_indices = np.arange(condition2_trials.shape[0])

        for train_idx, test_idx in kf.split(condition1_indices, condition2_indices):
            train_condition1 = condition1_trials[train_idx]
            train_condition2 = condition2_trials[train_idx]
            test_condition1 = condition1_trials[test_idx]
            test_condition2 = condition2_trials[test_idx]

            # Group the training trials into 3 groups by averaging the first dimension
            train_condition1_averaged_1 = np.array_split(train_condition1, 3, axis=0)
            train_condition1_averaged_2 = [np.mean(group, axis=0) for group in train_condition1_averaged_1]
            train_condition1_averaged = np.array(train_condition1_averaged_2)

            train_condition2_averaged_1 = np.array_split(train_condition2, 3, axis=0)
            train_condition2_averaged_2 = [np.mean(group, axis=0) for group in train_condition2_averaged_1]
            train_condition2_averaged = np.array(train_condition2_averaged_2)

            # Keep the test trials as a single group
            test_condition1_averaged = np.mean(test_condition1, axis=0, keepdims=True)
            test_condition2_averaged = np.mean(test_condition2, axis=0, keepdims=True)

            # Stack the training data along the first axis
            train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
            train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

            test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
            test_labels = np.array([1] * test_condition1_averaged.shape[0] + [0] * test_condition2_averaged.shape[0])

            # Replace NaN values with zeros in train and test data
            # train_data = np.nan_to_num(train_data, copy=False)
            # test_data = np.nan_to_num(test_data, copy=False)

            clf = SVC(kernel='linear')
            clf.fit(train_data, train_labels)
            accuracy = clf.score(test_data, test_labels)
            accuracies.append(accuracy)

print(f"Mean accuracy for {condition1} vs {condition2}: {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [01:03<00:00,  3.19s/it]

Mean accuracy for pleasant vs neutral: 0.8277


In [7]:
accuracies

[1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 1.0,
 1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 1.0,
 0.5,
 0.5,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 0.5,
 1.0,
 1.0,
 0.5,
 0.0,
 1.0,
 1.0,
 1.0,
 0.5,
 0.5,
 1.0,
 1.0,
 0.5,
 0.5,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 0.5,
 0.5,
 0.5,
 1.0,
 1.0,
 1.0,
 0.5,
 0.5,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 0.5,
 1.0,
 1.0,
 1.0,
 0.5,
 1.0,
 0.5,
 0.5,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 0.5,
 1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 1.0,
 0.5,
 1.0,
 1.0,
 0.5,
 1.0,
 0.5,
 0.5,
 1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 0.5,
 0.5,
 1.0,
 1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 0.0,
 1.0,
 1.0,
 0.5,
 1.0,
 1.0,
 0.5,
 0.0,
 0.5,
 0.5,
 1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 1.0,
 0.5

In [60]:
print(np.mean(accuracies[:80]), np.mean(accuracies[80:160]), np.mean(accuracies[160:240]),  np.mean(accuracies[240:320]), np.mean(accuracies[320:400]), np.mean(accuracies[400:480]), np.mean(accuracies[480:560]))

0.8625 0.81875 0.63125 0.96875 0.76875 0.8125 0.93125


In [61]:
# Specify the two conditions you want to use for decoding
condition1 = 'unpleasant'
condition2 = 'neutral'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]

    # Calculate z-score for condition1_trials and condition2_trials
    # condition1_trials = zscore(condition1_trials, axis=1)
    # condition2_trials = zscore(condition2_trials, axis=1)

    for repeat in tqdm(range(20)):
        kf = KFold(n_splits=4, shuffle=True, random_state=repeat)
        condition1_indices = np.arange(condition1_trials.shape[0])
        condition2_indices = np.arange(condition2_trials.shape[0])

        for train_idx, test_idx in kf.split(condition1_indices, condition2_indices):
            train_condition1 = condition1_trials[train_idx]
            train_condition2 = condition2_trials[train_idx]
            test_condition1 = condition1_trials[test_idx]
            test_condition2 = condition2_trials[test_idx]

            # Group the training trials into 3 groups by averaging the first dimension
            train_condition1_averaged_1 = np.array_split(train_condition1, 3, axis=0)
            train_condition1_averaged_2 = [np.mean(group, axis=0) for group in train_condition1_averaged_1]
            train_condition1_averaged = np.array(train_condition1_averaged_2)

            train_condition2_averaged_1 = np.array_split(train_condition2, 3, axis=0)
            train_condition2_averaged_2 = [np.mean(group, axis=0) for group in train_condition2_averaged_1]
            train_condition2_averaged = np.array(train_condition2_averaged_2)

            # Keep the test trials as a single group
            test_condition1_averaged = np.mean(test_condition1, axis=0, keepdims=True)
            test_condition2_averaged = np.mean(test_condition2, axis=0, keepdims=True)

            # Stack the training data along the first axis
            train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
            train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

            test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
            test_labels = np.array([1] * test_condition1_averaged.shape[0] + [0] * test_condition2_averaged.shape[0])

            # Replace NaN values with zeros in train and test data
            # train_data = np.nan_to_num(train_data, copy=False)
            # test_data = np.nan_to_num(test_data, copy=False)

            clf = SVC(kernel='linear')
            clf.fit(train_data, train_labels)
            accuracy = clf.score(test_data, test_labels)
            accuracies.append(accuracy)

print(f"Mean accuracy for {condition1} vs {condition2}: {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [01:08<00:00,  3.41s/it]

Mean accuracy for unpleasant vs neutral: 0.6268


In [62]:
print(np.mean(accuracies[:80]), np.mean(accuracies[80:160]), np.mean(accuracies[160:240]),  np.mean(accuracies[240:320]), np.mean(accuracies[320:400]), np.mean(accuracies[400:480]), np.mean(accuracies[480:560]))

0.6 0.525 0.76875 0.7125 0.49375 0.675 0.6125


In [9]:
# Specify the two conditions you want to use for decoding
condition1 = 'unpleasant'
condition2 = 'neutral'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]

    # Calculate z-score for condition1_trials and condition2_trials
    # condition1_trials = zscore(condition1_trials, axis=1)
    # condition2_trials = zscore(condition2_trials, axis=1)

    for repeat in tqdm(range(20)):
        kf = KFold(n_splits=10, shuffle=True, random_state=repeat)
        condition1_indices = np.arange(condition1_trials.shape[0])
        condition2_indices = np.arange(condition2_trials.shape[0])

        for train_idx, test_idx in kf.split(condition1_indices, condition2_indices):
            train_condition1 = condition1_trials[train_idx]
            train_condition2 = condition2_trials[train_idx]
            test_condition1 = condition1_trials[test_idx]
            test_condition2 = condition2_trials[test_idx]

            # Group the training trials into 3 groups by averaging the first dimension
            train_condition1_averaged_1 = np.array_split(train_condition1, 10, axis=0)
            train_condition1_averaged_2 = [np.mean(group, axis=0) for group in train_condition1_averaged_1]
            train_condition1_averaged = np.array(train_condition1_averaged_2)

            train_condition2_averaged_1 = np.array_split(train_condition2, 10, axis=0)
            train_condition2_averaged_2 = [np.mean(group, axis=0) for group in train_condition2_averaged_1]
            train_condition2_averaged = np.array(train_condition2_averaged_2)

            # Keep the test trials as a single group
            test_condition1_averaged = np.mean(test_condition1, axis=0, keepdims=True)
            test_condition2_averaged = np.mean(test_condition2, axis=0, keepdims=True)

            # Stack the training data along the first axis
            train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
            train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

            test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
            test_labels = np.array([1] * test_condition1_averaged.shape[0] + [0] * test_condition2_averaged.shape[0])

            # Replace NaN values with zeros in train and test data
            # train_data = np.nan_to_num(train_data, copy=False)
            # test_data = np.nan_to_num(test_data, copy=False)

            clf = SVC(kernel='linear')
            clf.fit(train_data, train_labels)
            accuracy = clf.score(test_data, test_labels)
            accuracies.append(accuracy)

print(f"Mean accuracy for {condition1} vs {condition2}: {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [02:58<00:00,  8.92s/it]

Mean accuracy for unpleasant vs neutral: 0.6064


In [11]:
# Specify the two conditions you want to use for decoding
condition1 = 'unpleasant'
condition2 = 'neutral'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]

    # Calculate z-score for condition1_trials and condition2_trials
    # condition1_trials = zscore(condition1_trials, axis=1)
    # condition2_trials = zscore(condition2_trials, axis=1)

    for repeat in tqdm(range(20)):
        kf = KFold(n_splits=5, shuffle=True, random_state=repeat)
        condition1_indices = np.arange(condition1_trials.shape[0])
        condition2_indices = np.arange(condition2_trials.shape[0])

        for train_idx, test_idx in kf.split(condition1_indices, condition2_indices):
            train_condition1 = condition1_trials[train_idx]
            train_condition2 = condition2_trials[train_idx]
            test_condition1 = condition1_trials[test_idx]
            test_condition2 = condition2_trials[test_idx]

            # Group the training trials into 3 groups by averaging the first dimension
            train_condition1_averaged_1 = np.array_split(train_condition1, 5, axis=0)
            train_condition1_averaged_2 = [np.mean(group, axis=0) for group in train_condition1_averaged_1]
            train_condition1_averaged = np.array(train_condition1_averaged_2)

            train_condition2_averaged_1 = np.array_split(train_condition2, 5, axis=0)
            train_condition2_averaged_2 = [np.mean(group, axis=0) for group in train_condition2_averaged_1]
            train_condition2_averaged = np.array(train_condition2_averaged_2)

            test_condition1_averaged_1 = np.array_split(test_condition1, 5, axis=0)
            test_condition1_averaged_2 = [np.mean(group, axis=0) for group in test_condition1_averaged_1]
            test_condition1_averaged = np.array(test_condition1_averaged_2)

            test_condition2_averaged_1 = np.array_split(test_condition2, 5, axis=0)
            test_condition2_averaged_2 = [np.mean(group, axis=0) for group in test_condition2_averaged_1]
            test_condition2_averaged = np.array(test_condition2_averaged_2)


            # Stack the training data along the first axis
            train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
            train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

            test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
            test_labels = np.array([1] * len(test_condition1_averaged) + [0] * len(test_condition2_averaged))

            # Replace NaN values with zeros in train and test data
            # train_data = np.nan_to_num(train_data, copy=False)
            # test_data = np.nan_to_num(test_data, copy=False)

            clf = SVC(kernel='linear')
            clf.fit(train_data, train_labels)
            accuracy = clf.score(test_data, test_labels)
            accuracies.append(accuracy)

print(f"Mean accuracy for {condition1} vs {condition2}: {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [01:22<00:00,  4.12s/it]

Mean accuracy for unpleasant vs neutral: 0.5481


In [13]:
# Specify the two conditions you want to use for decoding
condition1 = 'unpleasant'
condition2 = 'neutral'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]

    # Calculate z-score for condition1_trials and condition2_trials
    # condition1_trials = zscore(condition1_trials, axis=1)
    # condition2_trials = zscore(condition2_trials, axis=1)

    for repeat in tqdm(range(20)):
        kf = KFold(n_splits=5, shuffle=True, random_state=repeat)
        condition1_indices = np.arange(condition1_trials.shape[0])
        condition2_indices = np.arange(condition2_trials.shape[0])

        for train_idx, test_idx in kf.split(condition1_indices, condition2_indices):
            train_condition1 = condition1_trials[train_idx]
            train_condition2 = condition2_trials[train_idx]
            test_condition1 = condition1_trials[test_idx]
            test_condition2 = condition2_trials[test_idx]

            # Group the training trials into 3 groups by averaging the first dimension
            train_condition1_averaged_1 = np.array_split(train_condition1, 16, axis=0)
            train_condition1_averaged_2 = [np.mean(group, axis=0) for group in train_condition1_averaged_1]
            train_condition1_averaged = np.array(train_condition1_averaged_2)

            train_condition2_averaged_1 = np.array_split(train_condition2, 16, axis=0)
            train_condition2_averaged_2 = [np.mean(group, axis=0) for group in train_condition2_averaged_1]
            train_condition2_averaged = np.array(train_condition2_averaged_2)

            test_condition1_averaged_1 = np.array_split(test_condition1, 4, axis=0)
            test_condition1_averaged_2 = [np.mean(group, axis=0) for group in test_condition1_averaged_1]
            test_condition1_averaged = np.array(test_condition1_averaged_2)

            test_condition2_averaged_1 = np.array_split(test_condition2,4, axis=0)
            test_condition2_averaged_2 = [np.mean(group, axis=0) for group in test_condition2_averaged_1]
            test_condition2_averaged = np.array(test_condition2_averaged_2)


            # Stack the training data along the first axis
            train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
            train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

            test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
            test_labels = np.array([1] * len(test_condition1_averaged) + [0] * len(test_condition2_averaged))

            # Replace NaN values with zeros in train and test data
            # train_data = np.nan_to_num(train_data, copy=False)
            # test_data = np.nan_to_num(test_data, copy=False)

            clf = SVC(kernel='linear')
            clf.fit(train_data, train_labels)
            accuracy = clf.score(test_data, test_labels)
            accuracies.append(accuracy)

print(f"Mean accuracy for {condition1} vs {condition2}: {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [01:50<00:00,  5.52s/it]

Mean accuracy for unpleasant vs neutral: 0.5764


In [63]:
# Specify the two conditions you want to use for decoding
condition1 = 'pleasantAI'
condition2 = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]

    # Calculate z-score for condition1_trials and condition2_trials
    # condition1_trials = zscore(condition1_trials, axis=1)
    # condition2_trials = zscore(condition2_trials, axis=1)

    for repeat in tqdm(range(20)):
        kf = KFold(n_splits=4, shuffle=True, random_state=repeat)
        condition1_indices = np.arange(condition1_trials.shape[0])
        condition2_indices = np.arange(condition2_trials.shape[0])

        for train_idx, test_idx in kf.split(condition1_indices, condition2_indices):
            train_condition1 = condition1_trials[train_idx]
            train_condition2 = condition2_trials[train_idx]
            test_condition1 = condition1_trials[test_idx]
            test_condition2 = condition2_trials[test_idx]

            # Group the training trials into 3 groups by averaging the first dimension
            train_condition1_averaged_1 = np.array_split(train_condition1, 3, axis=0)
            train_condition1_averaged_2 = [np.mean(group, axis=0) for group in train_condition1_averaged_1]
            train_condition1_averaged = np.array(train_condition1_averaged_2)

            train_condition2_averaged_1 = np.array_split(train_condition2, 3, axis=0)
            train_condition2_averaged_2 = [np.mean(group, axis=0) for group in train_condition2_averaged_1]
            train_condition2_averaged = np.array(train_condition2_averaged_2)

            # Keep the test trials as a single group
            test_condition1_averaged = np.mean(test_condition1, axis=0, keepdims=True)
            test_condition2_averaged = np.mean(test_condition2, axis=0, keepdims=True)

            # Stack the training data along the first axis
            train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
            train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

            test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
            test_labels = np.array([1] * test_condition1_averaged.shape[0] + [0] * test_condition2_averaged.shape[0])

            # Replace NaN values with zeros in train and test data
            # train_data = np.nan_to_num(train_data, copy=False)
            # test_data = np.nan_to_num(test_data, copy=False)

            clf = SVC(kernel='linear')
            clf.fit(train_data, train_labels)
            accuracy = clf.score(test_data, test_labels)
            accuracies.append(accuracy)

print(f"Mean accuracy for {condition1} vs {condition2}: {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [01:04<00:00,  3.21s/it]

Mean accuracy for pleasantAI vs neutralAI: 0.6661


In [64]:
print(np.mean(accuracies[:80]), np.mean(accuracies[80:160]), np.mean(accuracies[160:240]),  np.mean(accuracies[240:320]), np.mean(accuracies[320:400]), np.mean(accuracies[400:480]), np.mean(accuracies[480:560]))

0.71875 0.6 0.3625 0.775 0.85625 0.59375 0.75625


In [65]:
# Specify the two conditions you want to use for decoding
condition1 = 'unpleasantAI'
condition2 = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]

    # Calculate z-score for condition1_trials and condition2_trials
    # condition1_trials = zscore(condition1_trials, axis=1)
    # condition2_trials = zscore(condition2_trials, axis=1)

    for repeat in tqdm(range(20)):
        kf = KFold(n_splits=4, shuffle=True, random_state=repeat)
        condition1_indices = np.arange(condition1_trials.shape[0])
        condition2_indices = np.arange(condition2_trials.shape[0])

        for train_idx, test_idx in kf.split(condition1_indices, condition2_indices):
            train_condition1 = condition1_trials[train_idx]
            train_condition2 = condition2_trials[train_idx]
            test_condition1 = condition1_trials[test_idx]
            test_condition2 = condition2_trials[test_idx]

            # Group the training trials into 3 groups by averaging the first dimension
            train_condition1_averaged_1 = np.array_split(train_condition1, 3, axis=0)
            train_condition1_averaged_2 = [np.mean(group, axis=0) for group in train_condition1_averaged_1]
            train_condition1_averaged = np.array(train_condition1_averaged_2)

            train_condition2_averaged_1 = np.array_split(train_condition2, 3, axis=0)
            train_condition2_averaged_2 = [np.mean(group, axis=0) for group in train_condition2_averaged_1]
            train_condition2_averaged = np.array(train_condition2_averaged_2)

            # Keep the test trials as a single group
            test_condition1_averaged = np.mean(test_condition1, axis=0, keepdims=True)
            test_condition2_averaged = np.mean(test_condition2, axis=0, keepdims=True)

            # Stack the training data along the first axis
            train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
            train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

            test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
            test_labels = np.array([1] * test_condition1_averaged.shape[0] + [0] * test_condition2_averaged.shape[0])

            # Replace NaN values with zeros in train and test data
            # train_data = np.nan_to_num(train_data, copy=False)
            # test_data = np.nan_to_num(test_data, copy=False)

            clf = SVC(kernel='linear')
            clf.fit(train_data, train_labels)
            accuracy = clf.score(test_data, test_labels)
            accuracies.append(accuracy)

print(f"Mean accuracy for {condition1} vs {condition2}: {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [01:05<00:00,  3.26s/it]

Mean accuracy for unpleasantAI vs neutralAI: 0.6634


In [66]:
print(np.mean(accuracies[:80]), np.mean(accuracies[80:160]), np.mean(accuracies[160:240]),  np.mean(accuracies[240:320]), np.mean(accuracies[320:400]), np.mean(accuracies[400:480]), np.mean(accuracies[480:560]))

0.68125 0.775 0.6 0.5125 0.70625 0.66875 0.7


In [71]:
# Specify the two conditions you want to use for decoding
condition1 = 'unpleasantAI'
condition2 = 'unpleasant'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]

    # Calculate z-score for condition1_trials and condition2_trials
    # condition1_trials = zscore(condition1_trials, axis=1)
    # condition2_trials = zscore(condition2_trials, axis=1)

    for repeat in tqdm(range(20)):
        kf = KFold(n_splits=4, shuffle=True, random_state=repeat)
        condition1_indices = np.arange(condition1_trials.shape[0])
        condition2_indices = np.arange(condition2_trials.shape[0])

        for train_idx, test_idx in kf.split(condition1_indices, condition2_indices):
            train_condition1 = condition1_trials[train_idx]
            train_condition2 = condition2_trials[train_idx]
            test_condition1 = condition1_trials[test_idx]
            test_condition2 = condition2_trials[test_idx]

            # Group the training trials into 3 groups by averaging the first dimension
            train_condition1_averaged_1 = np.array_split(train_condition1, 3, axis=0)
            train_condition1_averaged_2 = [np.mean(group, axis=0) for group in train_condition1_averaged_1]
            train_condition1_averaged = np.array(train_condition1_averaged_2)

            train_condition2_averaged_1 = np.array_split(train_condition2, 3, axis=0)
            train_condition2_averaged_2 = [np.mean(group, axis=0) for group in train_condition2_averaged_1]
            train_condition2_averaged = np.array(train_condition2_averaged_2)

            # Keep the test trials as a single group
            test_condition1_averaged = np.mean(test_condition1, axis=0, keepdims=True)
            test_condition2_averaged = np.mean(test_condition2, axis=0, keepdims=True)

            # Stack the training data along the first axis
            train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
            train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

            test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
            test_labels = np.array([1] * test_condition1_averaged.shape[0] + [0] * test_condition2_averaged.shape[0])

            # Replace NaN values with zeros in train and test data
            # train_data = np.nan_to_num(train_data, copy=False)
            # test_data = np.nan_to_num(test_data, copy=False)

            clf = SVC(kernel='linear')
            clf.fit(train_data, train_labels)
            accuracy = clf.score(test_data, test_labels)
            accuracies.append(accuracy)

print(f"Mean accuracy for {condition1} vs {condition2}: {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [00:59<00:00,  2.98s/it]

Mean accuracy for unpleasantAI vs unpleasant: 0.5045


In [73]:
print(np.mean(accuracies[:80]), np.mean(accuracies[80:160]), np.mean(accuracies[160:240]),  np.mean(accuracies[240:320]), np.mean(accuracies[320:400]), np.mean(accuracies[400:480]), np.mean(accuracies[480:560]))

0.4875 0.4 0.50625 0.7625 0.3 0.65625 0.41875


In [68]:
# Specify the two conditions you want to use for decoding
condition1 = 'pleasantAI'
condition2 = 'pleasant'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]

    # Calculate z-score for condition1_trials and condition2_trials
    # condition1_trials = zscore(condition1_trials, axis=1)
    # condition2_trials = zscore(condition2_trials, axis=1)

    for repeat in tqdm(range(20)):
        kf = KFold(n_splits=4, shuffle=True, random_state=repeat)
        condition1_indices = np.arange(condition1_trials.shape[0])
        condition2_indices = np.arange(condition2_trials.shape[0])

        for train_idx, test_idx in kf.split(condition1_indices, condition2_indices):
            train_condition1 = condition1_trials[train_idx]
            train_condition2 = condition2_trials[train_idx]
            test_condition1 = condition1_trials[test_idx]
            test_condition2 = condition2_trials[test_idx]

            # Group the training trials into 3 groups by averaging the first dimension
            train_condition1_averaged_1 = np.array_split(train_condition1, 3, axis=0)
            train_condition1_averaged_2 = [np.mean(group, axis=0) for group in train_condition1_averaged_1]
            train_condition1_averaged = np.array(train_condition1_averaged_2)

            train_condition2_averaged_1 = np.array_split(train_condition2, 3, axis=0)
            train_condition2_averaged_2 = [np.mean(group, axis=0) for group in train_condition2_averaged_1]
            train_condition2_averaged = np.array(train_condition2_averaged_2)

            # Keep the test trials as a single group
            test_condition1_averaged = np.mean(test_condition1, axis=0, keepdims=True)
            test_condition2_averaged = np.mean(test_condition2, axis=0, keepdims=True)

            # Stack the training data along the first axis
            train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
            train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

            test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
            test_labels = np.array([1] * test_condition1_averaged.shape[0] + [0] * test_condition2_averaged.shape[0])

            # Replace NaN values with zeros in train and test data
            # train_data = np.nan_to_num(train_data, copy=False)
            # test_data = np.nan_to_num(test_data, copy=False)

            clf = SVC(kernel='linear')
            clf.fit(train_data, train_labels)
            accuracy = clf.score(test_data, test_labels)
            accuracies.append(accuracy)

print(f"Mean accuracy for {condition1} vs {condition2}: {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [01:01<00:00,  3.05s/it]

Mean accuracy for pleasantAI vs pleasant: 0.7134


In [70]:
print(np.mean(accuracies[:80]), np.mean(accuracies[80:160]), np.mean(accuracies[160:240]),  np.mean(accuracies[240:320]), np.mean(accuracies[320:400]), np.mean(accuracies[400:480]), np.mean(accuracies[480:560]))

0.73125 0.61875 0.5 0.85625 0.7625 0.75625 0.76875


In [74]:
# Specify the two conditions you want to use for decoding
condition1 = 'neutralAI'
condition2 = 'neutral'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]

    # Calculate z-score for condition1_trials and condition2_trials
    # condition1_trials = zscore(condition1_trials, axis=1)
    # condition2_trials = zscore(condition2_trials, axis=1)

    for repeat in tqdm(range(20)):
        kf = KFold(n_splits=4, shuffle=True, random_state=repeat)
        condition1_indices = np.arange(condition1_trials.shape[0])
        condition2_indices = np.arange(condition2_trials.shape[0])

        for train_idx, test_idx in kf.split(condition1_indices, condition2_indices):
            train_condition1 = condition1_trials[train_idx]
            train_condition2 = condition2_trials[train_idx]
            test_condition1 = condition1_trials[test_idx]
            test_condition2 = condition2_trials[test_idx]

            # Group the training trials into 3 groups by averaging the first dimension
            train_condition1_averaged_1 = np.array_split(train_condition1, 3, axis=0)
            train_condition1_averaged_2 = [np.mean(group, axis=0) for group in train_condition1_averaged_1]
            train_condition1_averaged = np.array(train_condition1_averaged_2)

            train_condition2_averaged_1 = np.array_split(train_condition2, 3, axis=0)
            train_condition2_averaged_2 = [np.mean(group, axis=0) for group in train_condition2_averaged_1]
            train_condition2_averaged = np.array(train_condition2_averaged_2)

            # Keep the test trials as a single group
            test_condition1_averaged = np.mean(test_condition1, axis=0, keepdims=True)
            test_condition2_averaged = np.mean(test_condition2, axis=0, keepdims=True)

            # Stack the training data along the first axis
            train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
            train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

            test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
            test_labels = np.array([1] * test_condition1_averaged.shape[0] + [0] * test_condition2_averaged.shape[0])

            # Replace NaN values with zeros in train and test data
            # train_data = np.nan_to_num(train_data, copy=False)
            # test_data = np.nan_to_num(test_data, copy=False)

            clf = SVC(kernel='linear')
            clf.fit(train_data, train_labels)
            accuracy = clf.score(test_data, test_labels)
            accuracies.append(accuracy)

print(f"Mean accuracy for {condition1} vs {condition2}: {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [01:00<00:00,  3.05s/it]

Mean accuracy for neutralAI vs neutral: 0.5473


In [75]:
print(np.mean(accuracies[:80]), np.mean(accuracies[80:160]), np.mean(accuracies[160:240]),  np.mean(accuracies[240:320]), np.mean(accuracies[320:400]), np.mean(accuracies[400:480]), np.mean(accuracies[480:560]))

0.58125 0.51875 0.4125 0.475 0.54375 0.83125 0.46875


# Cross decoding

## All trials

PL-NT

In [79]:
data.shape

(7, 600, 79, 95, 79)

In [81]:
# Specify the two conditions you want to use for decoding
condition1 = 'pleasant'
condition2 = 'neutral'
condition1_cross = 'pleasantAI'
condition2_cross = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)
condition1_cross_idx = conditions.index(condition1_cross)
condition2_cross_idx = conditions.index(condition2_cross)

accuracies = []

for subject in tqdm(range(data.shape[0])):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]
    condition1_cross_trials = subject_data[condition1_cross_idx * 100:(condition1_cross_idx + 1) * 100]
    condition2_cross_trials = subject_data[condition2_cross_idx * 100:(condition2_cross_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]
    condition1_cross_trials = condition1_cross_trials[:,~np.isnan(condition1_cross_trials[0, :])]
    condition2_cross_trials = condition2_cross_trials[:,~np.isnan(condition2_cross_trials[0, :])]

    # Stack the training data along the first axis
    train_data = np.vstack((condition1_trials, condition2_trials))
    train_labels = np.array([1] * len(condition1_trials) + [0] * len(condition2_trials))

    test_data = np.vstack((condition1_cross_trials, condition2_cross_trials))
    test_labels = np.array([1] * len(condition1_cross_trials) + [0] * len(condition2_cross_trials))

    # Replace NaN values with zeros in train and test data
    # train_data = np.nan_to_num(train_data, copy=False)
    # test_data = np.nan_to_num(test_data, copy=False)

    clf = SVC(kernel='linear')
    clf.fit(train_data, train_labels)
    accuracy = clf.score(test_data, test_labels)
    accuracies.append(accuracy)

print(f"Mean accuracy for crossdecoding, trained on {condition1} vs {condition2} and tested on {condition1_cross} vs {condition2_cross}: \n {np.mean(accuracies):.4f}")

Mean accuracy for crossdecoding, trained on pleasant vs neutral and tested on pleasantAI vs neutralAI: 
 0.5950


UP-NT

In [84]:
# Specify the two conditions you want to use for decoding
condition1 = 'unpleasant'
condition2 = 'neutral'
condition1_cross = 'unpleasantAI'
condition2_cross = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)
condition1_cross_idx = conditions.index(condition1_cross)
condition2_cross_idx = conditions.index(condition2_cross)

accuracies = []

for subject in tqdm(range(data.shape[0])):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]
    condition1_cross_trials = subject_data[condition1_cross_idx * 100:(condition1_cross_idx + 1) * 100]
    condition2_cross_trials = subject_data[condition2_cross_idx * 100:(condition2_cross_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]
    condition1_cross_trials = condition1_cross_trials[:,~np.isnan(condition1_cross_trials[0, :])]
    condition2_cross_trials = condition2_cross_trials[:,~np.isnan(condition2_cross_trials[0, :])]

    # Stack the training data along the first axis
    train_data = np.vstack((condition1_trials, condition2_trials))
    train_labels = np.array([1] * len(condition1_trials) + [0] * len(condition2_trials))

    test_data = np.vstack((condition1_cross_trials, condition2_cross_trials))
    test_labels = np.array([1] * len(condition1_cross_trials) + [0] * len(condition2_cross_trials))

    # Replace NaN values with zeros in train and test data
    # train_data = np.nan_to_num(train_data, copy=False)
    # test_data = np.nan_to_num(test_data, copy=False)

    clf = SVC(kernel='linear')
    clf.fit(train_data, train_labels)
    accuracy = clf.score(test_data, test_labels)
    accuracies.append(accuracy)

print(f"Mean accuracy for crossdecoding, trained on {condition1} vs {condition2} and tested on {condition1_cross} vs {condition2_cross}: \n {np.mean(accuracies):.4f}")

  0%|          | 0/7 [00:00<?, ?it/s]

100%|██████████| 7/7 [01:49<00:00, 15.62s/it]

Mean accuracy for crossdecoding, trained on unpleasant vs neutral and tested on unpleasantAI vs neutralAI: 
 0.5743


## Average trials

PL-NT

In [95]:
# Specify the two conditions you want to use for decoding
condition1 = 'pleasant'
condition2 = 'neutral'
condition1_cross = 'pleasantAI'
condition2_cross = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)
condition1_cross_idx = conditions.index(condition1_cross)
condition2_cross_idx = conditions.index(condition2_cross)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]
    condition1_cross_trials = subject_data[condition1_cross_idx * 100:(condition1_cross_idx + 1) * 100]
    condition2_cross_trials = subject_data[condition2_cross_idx * 100:(condition2_cross_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]
    condition1_cross_trials = condition1_cross_trials[:,~np.isnan(condition1_cross_trials[0, :])]
    condition2_cross_trials = condition2_cross_trials[:,~np.isnan(condition2_cross_trials[0, :])]

    for repeat in tqdm(range(20)):
        train_indices = np.random.permutation(condition1_trials.shape[0])
        group_indices = np.array_split(train_indices, 4)
    
        # Average trials within each group
        train_condition1_averaged = np.array([np.mean(condition1_trials[group_idx, :], axis=0) for group_idx in group_indices])
        train_condition2_averaged = np.array([np.mean(condition2_trials[group_idx, :], axis=0) for group_idx in group_indices])

        # Stack the training data along the first axis
        train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
        train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

        # Keep the test trials as a single group
        test_data = np.vstack((np.mean(condition1_cross_trials, axis=0, keepdims=True),
                                np.mean(condition2_cross_trials, axis=0, keepdims=True)))
        test_labels = np.array([1, 0])

        # Replace NaN values with zeros in train and test data
        # train_data = np.nan_to_num(train_data, copy=False)
        # test_data = np.nan_to_num(test_data, copy=False)

        clf = SVC(kernel='linear')
        clf.fit(train_data, train_labels)
        accuracy = clf.score(test_data, test_labels)
        accuracies.append(accuracy)

print(f"Mean accuracy for crossdecoding, trained on {condition1} vs {condition2} and tested on {condition1_cross} vs {condition2_cross}: \n {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [00:17<00:00,  1.16it/s]

Mean accuracy for crossdecoding, trained on pleasant vs neutral and tested on pleasantAI vs neutralAI: 
 0.5536


UP-NT

In [96]:
# Specify the two conditions you want to use for decoding
condition1 = 'unpleasant'
condition2 = 'neutral'
condition1_cross = 'unpleasantAI'
condition2_cross = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)
condition1_cross_idx = conditions.index(condition1_cross)
condition2_cross_idx = conditions.index(condition2_cross)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]
    condition1_cross_trials = subject_data[condition1_cross_idx * 100:(condition1_cross_idx + 1) * 100]
    condition2_cross_trials = subject_data[condition2_cross_idx * 100:(condition2_cross_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]
    condition1_cross_trials = condition1_cross_trials[:,~np.isnan(condition1_cross_trials[0, :])]
    condition2_cross_trials = condition2_cross_trials[:,~np.isnan(condition2_cross_trials[0, :])]

    for repeat in tqdm(range(20)):
        train_indices = np.random.permutation(condition1_trials.shape[0])
        group_indices = np.array_split(train_indices, 4)
    
        # Average trials within each group
        train_condition1_averaged = np.array([np.mean(condition1_trials[group_idx, :], axis=0) for group_idx in group_indices])
        train_condition2_averaged = np.array([np.mean(condition2_trials[group_idx, :], axis=0) for group_idx in group_indices])
        # Stack the training data along the first axis
        train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
        train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

        # Keep the test trials as a single group
        test_data = np.vstack((np.mean(condition1_cross_trials, axis=0, keepdims=True),
                                np.mean(condition2_cross_trials, axis=0, keepdims=True)))
        test_labels = np.array([1, 0])

        # Replace NaN values with zeros in train and test data
        # train_data = np.nan_to_num(train_data, copy=False)
        # test_data = np.nan_to_num(test_data, copy=False)

        clf = SVC(kernel='linear')
        clf.fit(train_data, train_labels)
        accuracy = clf.score(test_data, test_labels)
        accuracies.append(accuracy)

print(f"Mean accuracy for crossdecoding, trained on {condition1} vs {condition2} and tested on {condition1_cross} vs {condition2_cross}: \n {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [00:18<00:00,  1.09it/s]

Mean accuracy for crossdecoding, trained on unpleasant vs neutral and tested on unpleasantAI vs neutralAI: 
 0.8786


In [99]:
# Specify the two conditions you want to use for decoding
condition1 = 'pleasant'
condition2 = 'neutral'
condition1_cross = 'pleasantAI'
condition2_cross = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)
condition1_cross_idx = conditions.index(condition1_cross)
condition2_cross_idx = conditions.index(condition2_cross)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]
    condition1_cross_trials = subject_data[condition1_cross_idx * 100:(condition1_cross_idx + 1) * 100]
    condition2_cross_trials = subject_data[condition2_cross_idx * 100:(condition2_cross_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]
    condition1_cross_trials = condition1_cross_trials[:,~np.isnan(condition1_cross_trials[0, :])]
    condition2_cross_trials = condition2_cross_trials[:,~np.isnan(condition2_cross_trials[0, :])]

    for repeat in tqdm(range(20)):
        train_indices = np.random.permutation(condition1_trials.shape[0])
        group_indices = np.array_split(train_indices, 4)
    
        # Average trials within each group
        train_condition1_averaged = np.array([np.mean(condition1_trials[group_idx, :], axis=0) for group_idx in group_indices])
        train_condition2_averaged = np.array([np.mean(condition2_trials[group_idx, :], axis=0) for group_idx in group_indices])
        # Stack the training data along the first axis
        train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
        train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

        # Average trials within each group
        test_condition1_averaged = np.array([np.mean(condition1_cross_trials[group_idx, :], axis=0) for group_idx in group_indices])
        test_condition2_averaged = np.array([np.mean(condition2_cross_trials[group_idx, :], axis=0) for group_idx in group_indices])
        # Stack the training data along the first axis
        test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
        test_labels = np.array([1] * len(test_condition1_averaged) + [0] * len(test_condition2_averaged))



        # Replace NaN values with zeros in train and test data
        # train_data = np.nan_to_num(train_data, copy=False)
        # test_data = np.nan_to_num(test_data, copy=False)

        clf = SVC(kernel='linear')
        clf.fit(train_data, train_labels)
        accuracy = clf.score(test_data, test_labels)
        accuracies.append(accuracy)

print(f"Mean accuracy for crossdecoding, trained on {condition1} vs {condition2} and tested on {condition1_cross} vs {condition2_cross}: \n {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [00:32<00:00,  1.61s/it]

Mean accuracy for crossdecoding, trained on pleasant vs neutral and tested on pleasantAI vs neutralAI: 
 0.6143


In [98]:
# Specify the two conditions you want to use for decoding
condition1 = 'unpleasant'
condition2 = 'neutral'
condition1_cross = 'unpleasantAI'
condition2_cross = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)
condition1_cross_idx = conditions.index(condition1_cross)
condition2_cross_idx = conditions.index(condition2_cross)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]
    condition1_cross_trials = subject_data[condition1_cross_idx * 100:(condition1_cross_idx + 1) * 100]
    condition2_cross_trials = subject_data[condition2_cross_idx * 100:(condition2_cross_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]
    condition1_cross_trials = condition1_cross_trials[:,~np.isnan(condition1_cross_trials[0, :])]
    condition2_cross_trials = condition2_cross_trials[:,~np.isnan(condition2_cross_trials[0, :])]

    for repeat in tqdm(range(20)):
        train_indices = np.random.permutation(condition1_trials.shape[0])
        group_indices = np.array_split(train_indices, 4)
    
        # Average trials within each group
        train_condition1_averaged = np.array([np.mean(condition1_trials[group_idx, :], axis=0) for group_idx in group_indices])
        train_condition2_averaged = np.array([np.mean(condition2_trials[group_idx, :], axis=0) for group_idx in group_indices])
        # Stack the training data along the first axis
        train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
        train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

        # Average trials within each group
        test_condition1_averaged = np.array([np.mean(condition1_cross_trials[group_idx, :], axis=0) for group_idx in group_indices])
        test_condition2_averaged = np.array([np.mean(condition2_cross_trials[group_idx, :], axis=0) for group_idx in group_indices])
        # Stack the training data along the first axis
        test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
        test_labels = np.array([1] * len(test_condition1_averaged) + [0] * len(test_condition2_averaged))



        # Replace NaN values with zeros in train and test data
        # train_data = np.nan_to_num(train_data, copy=False)
        # test_data = np.nan_to_num(test_data, copy=False)

        clf = SVC(kernel='linear')
        clf.fit(train_data, train_labels)
        accuracy = clf.score(test_data, test_labels)
        accuracies.append(accuracy)

print(f"Mean accuracy for crossdecoding, trained on {condition1} vs {condition2} and tested on {condition1_cross} vs {condition2_cross}: \n {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [00:33<00:00,  1.66s/it]

Mean accuracy for crossdecoding, trained on unpleasant vs neutral and tested on unpleasantAI vs neutralAI: 
 0.7384


## Testing

train and test data are different cateegories, expect 50% accuracy

In [100]:
# Specify the two conditions you want to use for decoding
condition1 = 'unpleasant'
condition2 = 'neutral'
condition1_cross = 'pleasantAI'
condition2_cross = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)
condition1_cross_idx = conditions.index(condition1_cross)
condition2_cross_idx = conditions.index(condition2_cross)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]
    condition1_cross_trials = subject_data[condition1_cross_idx * 100:(condition1_cross_idx + 1) * 100]
    condition2_cross_trials = subject_data[condition2_cross_idx * 100:(condition2_cross_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]
    condition1_cross_trials = condition1_cross_trials[:,~np.isnan(condition1_cross_trials[0, :])]
    condition2_cross_trials = condition2_cross_trials[:,~np.isnan(condition2_cross_trials[0, :])]

    for repeat in tqdm(range(20)):
        train_indices = np.random.permutation(condition1_trials.shape[0])
        group_indices = np.array_split(train_indices, 4)
    
        # Average trials within each group
        train_condition1_averaged = np.array([np.mean(condition1_trials[group_idx, :], axis=0) for group_idx in group_indices])
        train_condition2_averaged = np.array([np.mean(condition2_trials[group_idx, :], axis=0) for group_idx in group_indices])
        # Stack the training data along the first axis
        train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
        train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

        # Average trials within each group
        test_condition1_averaged = np.array([np.mean(condition1_cross_trials[group_idx, :], axis=0) for group_idx in group_indices])
        test_condition2_averaged = np.array([np.mean(condition2_cross_trials[group_idx, :], axis=0) for group_idx in group_indices])
        # Stack the training data along the first axis
        test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
        test_labels = np.array([1] * len(test_condition1_averaged) + [0] * len(test_condition2_averaged))



        # Replace NaN values with zeros in train and test data
        # train_data = np.nan_to_num(train_data, copy=False)
        # test_data = np.nan_to_num(test_data, copy=False)

        clf = SVC(kernel='linear')
        clf.fit(train_data, train_labels)
        accuracy = clf.score(test_data, test_labels)
        accuracies.append(accuracy)

print(f"Mean accuracy for crossdecoding, trained on {condition1} vs {condition2} and tested on {condition1_cross} vs {condition2_cross}: \n {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [00:32<00:00,  1.62s/it]

Mean accuracy for crossdecoding, trained on unpleasant vs neutral and tested on pleasantAI vs neutralAI: 
 0.6277


In [101]:
# Specify the two conditions you want to use for decoding
condition1 = 'pleasant'
condition2 = 'neutral'
condition1_cross = 'unpleasantAI'
condition2_cross = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)
condition1_cross_idx = conditions.index(condition1_cross)
condition2_cross_idx = conditions.index(condition2_cross)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]
    condition1_cross_trials = subject_data[condition1_cross_idx * 100:(condition1_cross_idx + 1) * 100]
    condition2_cross_trials = subject_data[condition2_cross_idx * 100:(condition2_cross_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]
    condition1_cross_trials = condition1_cross_trials[:,~np.isnan(condition1_cross_trials[0, :])]
    condition2_cross_trials = condition2_cross_trials[:,~np.isnan(condition2_cross_trials[0, :])]

    for repeat in tqdm(range(20)):
        train_indices = np.random.permutation(condition1_trials.shape[0])
        group_indices = np.array_split(train_indices, 4)
    
        # Average trials within each group
        train_condition1_averaged = np.array([np.mean(condition1_trials[group_idx, :], axis=0) for group_idx in group_indices])
        train_condition2_averaged = np.array([np.mean(condition2_trials[group_idx, :], axis=0) for group_idx in group_indices])
        # Stack the training data along the first axis
        train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
        train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

        # Average trials within each group
        test_condition1_averaged = np.array([np.mean(condition1_cross_trials[group_idx, :], axis=0) for group_idx in group_indices])
        test_condition2_averaged = np.array([np.mean(condition2_cross_trials[group_idx, :], axis=0) for group_idx in group_indices])
        # Stack the training data along the first axis
        test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
        test_labels = np.array([1] * len(test_condition1_averaged) + [0] * len(test_condition2_averaged))



        # Replace NaN values with zeros in train and test data
        # train_data = np.nan_to_num(train_data, copy=False)
        # test_data = np.nan_to_num(test_data, copy=False)

        clf = SVC(kernel='linear')
        clf.fit(train_data, train_labels)
        accuracy = clf.score(test_data, test_labels)
        accuracies.append(accuracy)

print(f"Mean accuracy for crossdecoding, trained on {condition1} vs {condition2} and tested on {condition1_cross} vs {condition2_cross}: \n {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [00:33<00:00,  1.67s/it]

Mean accuracy for crossdecoding, trained on pleasant vs neutral and tested on unpleasantAI vs neutralAI: 
 0.6670


In [102]:
# Specify the two conditions you want to use for decoding
condition1 = 'unpleasant'
condition2 = 'neutral'
condition1_cross = 'pleasantAI'
condition2_cross = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)
condition1_cross_idx = conditions.index(condition1_cross)
condition2_cross_idx = conditions.index(condition2_cross)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]
    condition1_cross_trials = subject_data[condition1_cross_idx * 100:(condition1_cross_idx + 1) * 100]
    condition2_cross_trials = subject_data[condition2_cross_idx * 100:(condition2_cross_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]
    condition1_cross_trials = condition1_cross_trials[:,~np.isnan(condition1_cross_trials[0, :])]
    condition2_cross_trials = condition2_cross_trials[:,~np.isnan(condition2_cross_trials[0, :])]

    for repeat in tqdm(range(20)):
        train_indices = np.random.permutation(condition1_trials.shape[0])
        group_indices = np.array_split(train_indices, 4)
    
        # Average trials within each group
        train_condition1_averaged = np.array([np.mean(condition1_trials[group_idx, :], axis=0) for group_idx in group_indices])
        train_condition2_averaged = np.array([np.mean(condition2_trials[group_idx, :], axis=0) for group_idx in group_indices])
        # Stack the training data along the first axis
        train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
        train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

        # Keep the test trials as a single group
        test_data = np.vstack((np.mean(condition1_cross_trials, axis=0, keepdims=True),
                                np.mean(condition2_cross_trials, axis=0, keepdims=True)))
        test_labels = np.array([1, 0])

        # Replace NaN values with zeros in train and test data
        # train_data = np.nan_to_num(train_data, copy=False)
        # test_data = np.nan_to_num(test_data, copy=False)

        clf = SVC(kernel='linear')
        clf.fit(train_data, train_labels)
        accuracy = clf.score(test_data, test_labels)
        accuracies.append(accuracy)

print(f"Mean accuracy for crossdecoding, trained on {condition1} vs {condition2} and tested on {condition1_cross} vs {condition2_cross}: \n {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [00:17<00:00,  1.16it/s]

Mean accuracy for crossdecoding, trained on unpleasant vs neutral and tested on pleasantAI vs neutralAI: 
 0.8143


In [103]:
# Specify the two conditions you want to use for decoding
condition1 = 'pleasant'
condition2 = 'neutral'
condition1_cross = 'unpleasantAI'
condition2_cross = 'neutralAI'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)
condition1_cross_idx = conditions.index(condition1_cross)
condition2_cross_idx = conditions.index(condition2_cross)

accuracies = []

for subject in range(data.shape[0]):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]
    condition1_cross_trials = subject_data[condition1_cross_idx * 100:(condition1_cross_idx + 1) * 100]
    condition2_cross_trials = subject_data[condition2_cross_idx * 100:(condition2_cross_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]
    condition1_cross_trials = condition1_cross_trials[:,~np.isnan(condition1_cross_trials[0, :])]
    condition2_cross_trials = condition2_cross_trials[:,~np.isnan(condition2_cross_trials[0, :])]

    for repeat in tqdm(range(20)):
        train_indices = np.random.permutation(condition1_trials.shape[0])
        group_indices = np.array_split(train_indices, 4)
    
        # Average trials within each group
        train_condition1_averaged = np.array([np.mean(condition1_trials[group_idx, :], axis=0) for group_idx in group_indices])
        train_condition2_averaged = np.array([np.mean(condition2_trials[group_idx, :], axis=0) for group_idx in group_indices])
        # Stack the training data along the first axis
        train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
        train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

        # Keep the test trials as a single group
        test_data = np.vstack((np.mean(condition1_cross_trials, axis=0, keepdims=True),
                                np.mean(condition2_cross_trials, axis=0, keepdims=True)))
        test_labels = np.array([1, 0])

        # Replace NaN values with zeros in train and test data
        # train_data = np.nan_to_num(train_data, copy=False)
        # test_data = np.nan_to_num(test_data, copy=False)

        clf = SVC(kernel='linear')
        clf.fit(train_data, train_labels)
        accuracy = clf.score(test_data, test_labels)
        accuracies.append(accuracy)

print(f"Mean accuracy for crossdecoding, trained on {condition1} vs {condition2} and tested on {condition1_cross} vs {condition2_cross}: \n {np.mean(accuracies):.4f}")

100%|██████████| 20/20 [00:16<00:00,  1.20it/s]

Mean accuracy for crossdecoding, trained on pleasant vs neutral and tested on unpleasantAI vs neutralAI: 
 0.7393


In [106]:
# Specify the two conditions you want to use for decoding
condition1 = 'pleasant'
condition2 = 'neutral'
condition1_cross = 'unpleasant'
condition2_cross = 'neutral'

condition1_idx = conditions.index(condition1)
condition2_idx = conditions.index(condition2)
condition1_cross_idx = conditions.index(condition1_cross)
condition2_cross_idx = conditions.index(condition2_cross)

accuracies = []

for subject in tqdm(range(data.shape[0])):
    subject_data = data[subject]
    condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
    condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]
    condition1_cross_trials = subject_data[condition1_cross_idx * 100:(condition1_cross_idx + 1) * 100]
    condition2_cross_trials = subject_data[condition2_cross_idx * 100:(condition2_cross_idx + 1) * 100]

    condition1_trials = condition1_trials[:,~np.isnan(condition1_trials[0, :])]
    condition2_trials = condition2_trials[:,~np.isnan(condition2_trials[0, :])]
    condition1_cross_trials = condition1_cross_trials[:,~np.isnan(condition1_cross_trials[0, :])]
    condition2_cross_trials = condition2_cross_trials[:,~np.isnan(condition2_cross_trials[0, :])]

    # Stack the training data along the first axis
    train_data = np.vstack((condition1_trials, condition2_trials))
    train_labels = np.array([1] * len(condition1_trials) + [0] * len(condition2_trials))

    test_data = np.vstack((condition1_cross_trials, condition2_cross_trials))
    test_labels = np.array([1] * len(condition1_cross_trials) + [0] * len(condition2_cross_trials))

    # Replace NaN values with zeros in train and test data
    # train_data = np.nan_to_num(train_data, copy=False)
    # test_data = np.nan_to_num(test_data, copy=False)

    clf = SVC(kernel='linear')
    clf.fit(train_data, train_labels)
    accuracy = clf.score(test_data, test_labels)
    accuracies.append(accuracy)

print(f"Mean accuracy for crossdecoding, trained on {condition1} vs {condition2} and tested on {condition1_cross} vs {condition2_cross}: \n {np.mean(accuracies):.4f}")

  0%|          | 0/7 [00:00<?, ?it/s]

100%|██████████| 7/7 [01:41<00:00, 14.43s/it]

Mean accuracy for crossdecoding, trained on pleasant vs neutral and tested on unpleasant vs neutral: 
 0.7150


In [107]:
subject_data.shape

(600, 79, 95, 79)